# Week 3 – Unsupervised Learning & Model Evaluation

**YUVA Internship Project**

This notebook covers clustering techniques, dimensionality reduction, and supervised model evaluation using Scikit-learn.

### Topics
- K-Means clustering
- Hierarchical (Agglomerative) clustering
- Elbow method and silhouette score
- PCA dimensionality reduction
- Train/test evaluation
- Confusion matrix
- Accuracy, precision, recall and F1-score
- ROC-AUC
- 5-fold cross-validation
- Hyperparameter tuning with GridSearchCV


## 1. Objective

The objective is to understand clustering algorithms such as K-Means and Hierarchical Clustering, along with dimensionality reduction and model-evaluation concepts.

The project also demonstrates cross-validation, confusion matrix, precision, recall, F1-score, ROC-AUC, and hyperparameter tuning.


## 2. Tools and Technologies

- Python 3
- NumPy
- Pandas
- Scikit-learn
- Jupyter Notebook / Google Colab


In [ ]:
# Import libraries
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


## 3. Dataset 1 – Iris for Unsupervised Learning

The Iris dataset contains measurements for 150 flower samples. The clustering analysis uses the feature measurements without using the known class labels during training.

The original labels are retained only for reference; they are not supplied to K-Means or Hierarchical Clustering.


In [ ]:
# Load Iris data
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

iris_df = pd.DataFrame(X_iris, columns=iris.feature_names)
iris_df.head()


In [ ]:
# Standardize features before clustering
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

print("Original shape:", X_iris.shape)
print("Scaled shape:", X_iris_scaled.shape)


## 4. Choosing the Number of Clusters – Elbow Method

K-Means requires the number of clusters to be specified. The elbow method examines within-cluster sum of squares (inertia) for several values of K.

For this project, K=3 is used because the Iris dataset has three natural groups and the elbow/silhouette analysis supports a compact three-cluster solution.


In [ ]:
# Calculate K-Means inertia for several K values
inertias = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_iris_scaled)
    inertias.append(model.inertia_)

elbow_results = pd.DataFrame({
    "K": range(2, 7),
    "Inertia": inertias
})

elbow_results


## 5. K-Means Clustering

K-Means assigns observations to K clusters by minimizing the distance between observations and their assigned cluster centers.


In [ ]:
# Fit K-Means with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_iris_scaled)

kmeans_silhouette = silhouette_score(X_iris_scaled, kmeans_labels)

print("K-Means silhouette score:", round(kmeans_silhouette, 4))


## 6. Hierarchical (Agglomerative) Clustering

Agglomerative clustering starts with individual observations and progressively merges similar observations into larger groups. Ward linkage is used here.


In [ ]:
# Fit Agglomerative / Hierarchical clustering
hierarchical = AgglomerativeClustering(n_clusters=3, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(X_iris_scaled)

hierarchical_silhouette = silhouette_score(
    X_iris_scaled, hierarchical_labels
)

print("Hierarchical clustering silhouette score:",
      round(hierarchical_silhouette, 4))


## 7. Dimensionality Reduction with PCA

Principal Component Analysis (PCA) transforms the original features into a smaller number of components while retaining as much variance as possible.

Two principal components are created for visualization.


In [ ]:
# Reduce the standardized Iris data to two principal components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_iris_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by 2 components:",
      round(pca.explained_variance_ratio_.sum(), 4))


In [ ]:
# Create a compact PCA dataframe with K-Means cluster labels
pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "KMeans_Cluster": kmeans_labels,
    "Hierarchical_Cluster": hierarchical_labels
})

pca_df.head()


## 8. Clustering Comparison

Silhouette score ranges from -1 to 1. Higher values generally indicate that observations are better separated from other clusters while remaining close to their own cluster.

The scores below allow the two clustering approaches to be compared using the same standardized dataset.


In [ ]:
clustering_results = pd.DataFrame({
    "Model": ["K-Means", "Hierarchical (Agglomerative)"],
    "Silhouette Score": [
        kmeans_silhouette,
        hierarchical_silhouette
    ]
})

clustering_results.sort_values(
    "Silhouette Score", ascending=False
).reset_index(drop=True)


## 9. Dataset 2 – Breast Cancer for Model Evaluation

A separate classification example is used to demonstrate the model-evaluation requirements. The Breast Cancer Wisconsin dataset is split into training and testing sets using an 80:20 stratified split.

A Logistic Regression pipeline standardizes the features and then trains the classifier.


In [ ]:
# Load Breast Cancer data
breast_cancer = load_breast_cancer()
X_bc = breast_cancer.data
y_bc = breast_cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X_bc,
    y_bc,
    test_size=0.20,
    random_state=42,
    stratify=y_bc
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# Build and train a Logistic Regression pipeline
classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

classifier.fit(X_train, y_train)

predictions = classifier.predict(X_test)
probabilities = classifier.predict_proba(X_test)[:, 1]


## 10. Model Evaluation Metrics

The following metrics are calculated:

- **Accuracy:** proportion of correct predictions.
- **Precision:** proportion of predicted positive observations that are actually positive.
- **Recall:** proportion of actual positive observations that are identified.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures the classifier's ability to distinguish between the two classes across classification thresholds.


In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

evaluation_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

evaluation_results.round(4)


## 11. Confusion Matrix

A confusion matrix shows the counts of true negatives, false positives, false negatives, and true positives.


In [ ]:
# Display confusion matrix
cm = confusion_matrix(y_test, predictions)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

cm_df


## 12. Cross-Validation

Five-fold stratified cross-validation is used to evaluate the Logistic Regression pipeline across multiple train/validation splits.


In [ ]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    classifier,
    X_bc,
    y_bc,
    cv=cv,
    scoring="accuracy"
)

print("Cross-validation accuracy scores:",
      np.round(cv_scores, 4))
print("Mean CV accuracy:",
      round(cv_scores.mean(), 4))


## 13. Hyperparameter Tuning

GridSearchCV tests several values of Logistic Regression's regularization parameter `C` and two solvers.

ROC-AUC is used as the tuning metric, while five-fold stratified cross-validation is maintained.


In [ ]:
# Hyperparameter grid
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    estimator=classifier,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation ROC-AUC:",
      round(grid_search.best_score_, 4))


In [ ]:
# Evaluate the tuned model on the held-out test set
best_model = grid_search.best_estimator_

best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]

tuned_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, best_predictions),
        precision_score(y_test, best_predictions),
        recall_score(y_test, best_predictions),
        f1_score(y_test, best_predictions),
        roc_auc_score(y_test, best_probabilities)
    ]
})

tuned_results.round(4)


## 14. Conclusion

This project demonstrated unsupervised learning using K-Means and Hierarchical (Agglomerative) clustering, along with PCA for dimensionality reduction.

It also demonstrated practical model evaluation using a confusion matrix, accuracy, precision, recall, F1-score and ROC-AUC. Five-fold cross-validation and GridSearchCV were used to assess and tune a Logistic Regression model.

The project shows how clustering, dimensionality reduction, evaluation metrics, cross-validation and hyperparameter tuning fit into a complete machine-learning workflow.
